# 01 — Corpus Management

Load, inspect, add/remove verbs, download cross-linguistic treebanks, and extract semantic clauses.

**Run 00_Setup.ipynb first.**

In [ ]:
# ═══ Load the pre-computed English verb corpus ═══
import json
from pyodide.http import pyfetch
from collections import Counter

# Try loading from repo data
corpus = {}
try:
    resp = await pyfetch('../data/combined_corpus.json')
    corpus = json.loads(await resp.string())
    print(f'Loaded combined corpus: {len(corpus)} verbs')
except:
    try:
        resp = await pyfetch('../data/verbs.json')
        corpus = json.loads(await resp.string())
        print(f'Loaded verb inventory: {len(corpus)} verbs')
    except:
        print('No pre-computed corpus found.')
        print('Upload a CSV/JSON file or add verbs manually below.')

In [ ]:
# ═══ Corpus Statistics ═══
if corpus:
    # Count sources
    sources = Counter()
    for verb, info in corpus.items():
        if isinstance(info, dict):
            for src in info.get('sources', []):
                sources[src] += 1
    
    print(f'Total verbs: {len(corpus):,}')
    print(f'\nSources:')
    for src, count in sources.most_common():
        print(f'  {src}: {count:,}')
    
    # Sample verbs
    print(f'\nSample verbs:')
    for verb in list(corpus.keys())[:20]:
        info = corpus[verb]
        defn = info.get('definition', info.get('def', ''))[:60] if isinstance(info, dict) else str(info)[:60]
        print(f'  {verb:20s} {defn}')

In [ ]:
# ═══ Search the corpus ═══
search_term = 'create'  # Change this to search for different verbs

if corpus:
    matches = {v: info for v, info in corpus.items() if search_term in v}
    print(f'Found {len(matches)} verbs matching "{search_term}":')
    for v, info in list(matches.items())[:30]:
        defn = info.get('definition', '')[:70] if isinstance(info, dict) else ''
        print(f'  {v:25s} {defn}')

In [ ]:
# ═══ Add custom verbs ═══
# Add your own verbs to the corpus
custom_verbs = {
    # 'example_verb': {'definition': 'what this verb means', 'source': 'custom'},
}

if custom_verbs:
    corpus.update(custom_verbs)
    print(f'Added {len(custom_verbs)} custom verbs. Total: {len(corpus)}')
else:
    print('No custom verbs added. Edit the dict above to add your own.')

## Cross-Linguistic Verbs

Download Universal Dependencies treebanks and extract verb inventories from 30 typologically diverse languages.

In [ ]:
# ═══ Available UD Languages ═══
LANGUAGES = [
    ('Ancient_Greek',    'UD_Ancient_Greek-Perseus',    'grc', 'perseus',      'IE-Hellenic',     'ancient'),
    ('Latin',            'UD_Latin-Perseus',             'la',  'perseus',      'IE-Italic',       'ancient'),
    ('Classical_Chinese','UD_Classical_Chinese-Kyoto',   'lzh', 'kyoto',        'Sino-Tibetan',    'ancient'),
    ('Sanskrit',         'UD_Sanskrit-Vedic',            'sa',  'vedic',        'IE-Indo-Aryan',   'ancient'),
    ('Gothic',           'UD_Gothic-PROIEL',             'got', 'proiel',       'IE-Germanic',     'ancient'),
    ('English',          'UD_English-EWT',               'en',  'ewt',          'IE-Germanic',     'modern'),
    ('German',           'UD_German-GSD',                'de',  'gsd',          'IE-Germanic',     'modern'),
    ('Russian',          'UD_Russian-SynTagRus',         'ru',  'syntagrus',    'IE-Slavic',       'modern'),
    ('French',           'UD_French-GSD',                'fr',  'gsd',          'IE-Romance',      'modern'),
    ('Finnish',          'UD_Finnish-TDT',               'fi',  'tdt',          'Uralic',          'modern'),
    ('Japanese',         'UD_Japanese-GSD',              'ja',  'gsd',          'Japonic',         'modern'),
    ('Korean',           'UD_Korean-Kaist',              'ko',  'kaist',        'Koreanic',        'modern'),
    ('Arabic',           'UD_Arabic-PADT',               'ar',  'padt',         'Afro-Asiatic',    'modern'),
    ('Hindi',            'UD_Hindi-HDTB',                'hi',  'hdtb',         'IE-Indo-Aryan',   'modern'),
    ('Turkish',          'UD_Turkish-BOUN',              'tr',  'boun',         'Turkic',          'modern'),
    ('Indonesian',       'UD_Indonesian-GSD',            'id',  'gsd',          'Austronesian',    'modern'),
    ('Tamil',            'UD_Tamil-TTB',                 'ta',  'ttb',          'Dravidian',       'modern'),
    ('Basque',           'UD_Basque-BDT',               'eu',  'bdt',          'Isolate',         'modern'),
    ('Persian',          'UD_Persian-PerDT',            'fa',  'perdt',        'IE-Iranian',      'modern'),
    ('Coptic',           'UD_Coptic-Scriptorium',       'cop', 'scriptorium',  'Afro-Asiatic',    'ancient'),
]

print(f'{len(LANGUAGES)} languages available:')
print(f'{"Language":25s} {"Family":20s} {"Era"}')
print('-' * 55)
for name, tb, code, tbname, family, era in LANGUAGES:
    print(f'{name:25s} {family:20s} {era}')

In [ ]:
# ═══ Download a UD treebank and extract verbs ═══
# Change this to download a different language
TARGET_LANG = 'Latin'  

lang_info = next((l for l in LANGUAGES if l[0] == TARGET_LANG), None)
if not lang_info:
    print(f'Language {TARGET_LANG} not found')
else:
    name, treebank, code, tbname, family, era = lang_info
    UD_BASE = 'https://raw.githubusercontent.com/UniversalDependencies'
    
    combined_text = ''
    for split in ['train', 'dev', 'test']:
        url = f'{UD_BASE}/{treebank}/master/{code}_{tbname}-ud-{split}.conllu'
        print(f'Fetching {split}... ', end='')
        try:
            resp = await pyfetch(url)
            text = await resp.string()
            combined_text += text + '\n'
            print(f'{len(text):,} chars')
        except:
            print('not found (OK)')
    
    if combined_text:
        # Extract verbs using our parser
        try:
            from conllu_parser import extract_verbs
            verbs = extract_verbs(combined_text)
        except ImportError:
            # Inline extraction
            from collections import defaultdict
            verb_data = defaultdict(lambda: {'count': 0, 'forms': set()})
            for line in combined_text.split('\n'):
                line = line.strip()
                if not line or line.startswith('#'): continue
                fields = line.split('\t')
                if len(fields) < 10: continue
                if '-' in fields[0] or '.' in fields[0]: continue
                if fields[3] == 'VERB':
                    lemma = fields[2].lower().strip()
                    if lemma and lemma != '_':
                        verb_data[lemma]['count'] += 1
                        verb_data[lemma]['forms'].add(fields[1].lower())
            verbs = {k: {'count': v['count'], 'forms': sorted(list(v['forms']))[:10]} 
                     for k, v in verb_data.items()}
        
        print(f'\nExtracted {len(verbs)} unique verb lemmas from {name}')
        print(f'\nTop 20 by frequency:')
        for lemma, data in sorted(verbs.items(), key=lambda x: -x[1]['count'])[:20]:
            forms = ', '.join(data['forms'][:5])
            print(f'  {lemma:25s} freq={data["count"]:4d}  forms: {forms}')
    else:
        print(f'No data downloaded for {name}')

## Semantic Clause Extraction

Extract predicate-argument structures (subject-verb-object triples) from the downloaded CoNLL-U data.

In [ ]:
# ═══ Extract semantic clauses from the downloaded treebank ═══
if combined_text:
    try:
        from conllu_parser import extract_clauses, summarize_clauses
        clauses = extract_clauses(combined_text)
        summary = summarize_clauses(clauses)
    except ImportError:
        # Inline clause extraction
        print('Using inline clause extractor...')
        clauses = []
        sentences = []
        current = []
        for line in combined_text.split('\n'):
            line = line.strip()
            if not line:
                if current: sentences.append(current); current = []
                continue
            if line.startswith('#'): continue
            fields = line.split('\t')
            if len(fields) < 10 or '-' in fields[0] or '.' in fields[0]: continue
            try:
                current.append({'id': int(fields[0]), 'form': fields[1], 'lemma': fields[2],
                               'upos': fields[3], 'head': int(fields[6]) if fields[6] != '_' else 0,
                               'deprel': fields[7]})
            except: pass
        if current: sentences.append(current)
        
        for sent in sentences:
            for tok in sent:
                if tok['upos'] != 'VERB': continue
                clause = {'verb': tok['lemma'].lower(), 'subject': None, 'object': None}
                for t in sent:
                    if t['head'] == tok['id']:
                        dep = t['deprel'].split(':')[0]
                        if dep == 'nsubj': clause['subject'] = t['lemma'].lower()
                        elif dep == 'obj': clause['object'] = t['lemma'].lower()
                parts = [clause['subject'] or '', clause['verb'], clause['object'] or '']
                clause['full_text'] = ' '.join(p for p in parts if p)
                clauses.append(clause)
        
        summary = {
            'n_clauses': len(clauses),
            'n_unique_verbs': len(set(c['verb'] for c in clauses)),
            'sample_clauses': [c['full_text'] for c in clauses[:50]],
        }
    
    print(f'Extracted {summary["n_clauses"]} clauses with {summary["n_unique_verbs"]} unique verbs')
    print(f'\nSample predicate-argument structures:')
    for c in summary.get('sample_clauses', [])[:30]:
        print(f'  {c}')
else:
    print('No treebank data loaded. Run the download cell above first.')

In [ ]:
# ═══ Download the corpus as JSON ═══
# Creates a downloadable file in your browser

from js import Blob, URL, document

def download_json(data, filename):
    """Trigger a file download in the browser."""
    text = json.dumps(data, indent=2, ensure_ascii=False)
    blob = Blob.new([text], {'type': 'application/json'})
    url = URL.createObjectURL(blob)
    a = document.createElement('a')
    a.href = url
    a.download = filename
    a.click()
    URL.revokeObjectURL(url)
    print(f'Downloaded {filename} ({len(text):,} bytes)')

# Uncomment to download:
# download_json(corpus, 'eo_corpus.json')
print('Run download_json(corpus, "eo_corpus.json") to download')